## Stylized Facts of Cryptocurrencies

#### Importing packages and reading-in the dataframes

In [ ]:
#Perhaps it is necessary to reinstall the arch package
#%pip install arch

In [ ]:
## Importing packages
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

#--Checking heteroskedasticity
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
#--Downloading data
import pandas_datareader.data as web
import yfinance as yf
import datetime as dt
#--Plotting
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#-- Reading-in data from excel
project_root = Path.cwd() # Find the project root by searching upward for the Data folder
while not (project_root / "Data").exists() and project_root != project_root.parent:
    project_root = project_root.parent
project_root = str(project_root)

#-- Path to local Excel files
common_path = os.path.join(project_root, "Data") + os.sep
file_path_btc = common_path + "btcusd_data_20170817_20241231.xlsx"
file_path_eth = common_path + "ethusd_data_20170817_20241231.xlsx"
file_path_sp500 = common_path + "sp500_data_20170817_20241231.xlsx"
file_path_sol = common_path + "solusdt_data_20170817_20241231.xlsx"
file_path_bnb = common_path + "bnbusdt_data_20170817_20241011.xlsx"

#-- Read the Excel file
btc = pd.read_excel(file_path_btc)
eth = pd.read_excel(file_path_eth)
sp500 = pd.read_excel(file_path_sp500)
sol = pd.read_excel(file_path_sol)
bnb = pd.read_excel(file_path_bnb)

#-- Setting Open Time as index and making sure that the index of the dataframes is a datetimeindex
btc.set_index('Open Time', inplace=True)
eth.set_index('Open Time', inplace=True)
sp500.set_index('Date', inplace=True)
sol.set_index('Open Time', inplace=True)
bnb.set_index('Open Time', inplace=True)

In [ ]:
#-- Compute log returns
sp500['Log_Returns'] = np.log(sp500['Close'] / sp500['Close'].shift(1))
btc['Log_Returns'] = np.log(btc['Close'] / btc['Close'].shift(1))
eth['Log_Returns'] = np.log(eth['Close'] / eth['Close'].shift(1))
sol['Log_Returns'] = np.log(sol['Close'] / sol['Close'].shift(1))
bnb['Log_Returns'] = np.log(bnb['Close'] / bnb['Close'].shift(1))

#-- Dropping the first row since it contains a NaN
btc.dropna(inplace=True)
eth.dropna(inplace=True)
sp500.dropna(inplace=True)
sol.dropna(inplace=True)
bnb.dropna(inplace=True)

#### Fat-Tailed Distributions (Heavy-Tailed Returns)


In [ ]:
# --- 1. Fat-Tailed Distributions (Kurtosis) ---
kurt_btc = stats.kurtosis(btc['Log_Returns'], fisher=True)
kurt_eth = stats.kurtosis(eth['Log_Returns'], fisher=True)

print(f"Excess Kurtosis (BTC): {kurt_btc:.4f}")
print(f"Excess Kurtosis (ETH): {kurt_eth:.4f}")

**Fat-Tails (Kurtosis):** >10	Extremely fat-tailed, extreme outliers are common

#### Volatility Clustering (GARCH Effects)



In [ ]:
test_stat, p_value, _, _ = het_arch(btc['Log_Returns'])
print(f"ARCH test p-value BTC: {p_value:.20f}")
test_stat, p_value, _, _ = het_arch(eth['Log_Returns'])
print(f"ARCH test p-value ETH: {p_value:.20f}")
#Low p-value (< 0.05) → significant evidence of heteroskedasticity

**Volatility Clustering (ARCH Test):** If p-value < 0.05, cryptos exhibit clustered volatility.

In [ ]:
# --- 2. a. Volatility Clustering (ARCH Effect) ---

# We rescale the log returns by 10 to get a better model fit
arch_btc = arch_model(btc['Log_Returns']*10, vol='ARCH', p=1).fit(disp="off")
arch_eth = arch_model(eth['Log_Returns']*10, vol='ARCH', p=1).fit(disp="off")

print(f"ARCH Effect (BTC): p-value = {arch_btc.pvalues['omega']:.40f}")
print(f"ARCH Effect (ETH): p-value = {arch_eth.pvalues['omega']:.40f}")

In [ ]:
# --- 2. b. Volatility Clustering (GARCH effects) ---

# We now test for volatility depending on both past squared residuals AND past variances
# Fit GARCH(1,1) model to rescaled returns
garch_btc = arch_model(btc['Log_Returns']*10, vol='GARCH', p=1, q=1).fit(disp="off")
garch_eth = arch_model(eth['Log_Returns']*10, vol='GARCH', p=1, q=1).fit(disp="off")

# vol='GARCH':	Tells the model to use the GARCH volatility process
# p=1: Number of ARCH terms — impact of past squared residuals
# q=1: Number of GARCH terms — impact of past variances
# disp="off": Suppresses verbose output during fitting

# Extract p-values
btc_pvals = garch_btc.pvalues
eth_pvals = garch_eth.pvalues

# Extract alpha and beta values
alpha_btc = garch_btc.params['alpha[1]']
beta_btc = garch_btc.params['beta[1]']
sum_btc = alpha_btc + beta_btc

alpha_eth = garch_eth.params['alpha[1]']
beta_eth = garch_eth.params['beta[1]']
sum_eth = alpha_eth + beta_eth

# Print formatted p-values
print("BTC GARCH(1,1) p-values:")
print(f"omega    : {btc_pvals['omega']:.10f} < 0.05")
print(f"alpha[1] : {btc_pvals['alpha[1]']:.10f} < 0.05")
print(f"beta[1]  : {btc_pvals['beta[1]']:.10f} < 0.05")

# Print alpha + beta and interpretation
print(f"alpha + beta (BTC): {sum_btc:.4f} → {'close to 1: strong persistence' if 0.95 <= sum_btc <= 1.05 else 'not close to 1'}")

print("\nETH GARCH(1,1) p-values:")
print(f"omega    : {eth_pvals['omega']:.10f} > 0.05")
print(f"alpha[1] : {eth_pvals['alpha[1]']:.10f} < 0.05")
print(f"beta[1]  : {eth_pvals['beta[1]']:.10f} < 0.05")

# Print alpha + beta and interpretation
print(f"alpha + beta (ETH): {sum_eth:.4f} → {'close to 1: strong persistence' if 0.95 <= sum_eth <= 1.05 else 'not close to 1'}")

# This will show p-values for:
# omega: constant in variance equation
# alpha[1]: ARCH term (impact of recent shocks)
# beta[1]: GARCH term (impact of past volatility)

#### Asymmetric Returns (Positive Skew in Bull Markets, Negative Skew in Crashes)


In [ ]:
# --- 4. Skewness (Return Distributions) ---
skew_btc = stats.skew(btc['Log_Returns'])
skew_eth = stats.skew(eth['Log_Returns'])

print(f"Skewness (BTC): {skew_btc:.4f}")
print(f"Skewness (ETH): {skew_eth:.4f}")

#### Highly Non-Stationary Returns (No Strong Mean Reversion)


In [ ]:
# --- 3. Mean Reversion Test (ADF Test) ---
adf_btc = adfuller(btc['Close'])
adf_eth = adfuller(eth['Close'])

print(f"ADF Test (BTC): p-value = {adf_btc[1]:.4f} (Stationary if < 0.05)")
print(f"ADF Test (ETH): p-value = {adf_eth[1]:.4f} (Stationary if < 0.05)")

#### High Correlations Within Crypto Markets (But Changing Correlation with Equities)



In [ ]:
# --- 6. Correlation with Equities (Changing Over Time) ---
sp500['Log_Returns'] = np.log(sp500['Close'] / sp500['Close'].shift(1))
sp500.dropna(inplace=True)

#-- Merging the dataframes for easier correlation calculation
merged0 = pd.merge(btc['Log_Returns'], sp500['Log_Returns'], left_index=True, right_index=True, how='inner')
merged1 = pd.merge(btc['Log_Returns'], eth['Log_Returns'], left_index=True, right_index=True, how='inner')
merged3 = pd.merge(eth['Log_Returns'], sol['Log_Returns'], left_index=True, right_index=True, how='inner')
merged2 = pd.merge(eth['Log_Returns'], bnb['Log_Returns'], left_index=True, right_index=True, how='inner')
merged3 = pd.merge(eth['Log_Returns'], sol['Log_Returns'], left_index=True, right_index=True, how='inner')
merged4 = pd.merge(btc['Log_Returns'], bnb['Log_Returns'], left_index=True, right_index=True, how='inner')
merged5 = pd.merge(btc['Log_Returns'], sol['Log_Returns'], left_index=True, right_index=True, how='inner')
merged6 = pd.merge(bnb['Log_Returns'], sol['Log_Returns'], left_index=True, right_index=True, how='inner')

corr_btc_sp500 = merged0.corr().iloc[0, 1]
corr_btc_eth = merged1.corr().iloc[0, 1]
corr_eth_bnb = merged2.corr().iloc[0, 1]
corr_eth_sol = merged3.corr().iloc[0, 1]
corr_btc_bnb = merged4.corr().iloc[0, 1]
corr_btc_sol = merged5.corr().iloc[0, 1]
corr_bnb_sol = merged6.corr().iloc[0, 1]

print(f"\n Bitcoin-S&P 500 Correlation: {corr_btc_sp500:.4f}")
print(f"\n Bitcoin-Ethereum Correlation: {corr_btc_eth:.4f}")
print(f"\n Bitcoin-BinanceCoin Correlation: {corr_btc_bnb:.4f}")
print(f"\n Bitcoin-Solana Correlation: {corr_btc_sol:.4f}")
print(f"\n Ethereum-BinanceCoin Correlation: {corr_eth_bnb:.4f}")
print(f"\n Ethereum-Solana Correlation: {corr_eth_sol:.4f}")
print(f"\n BinanceCoin-Solana Correlation: {corr_bnb_sol:.4f}")

#### Strong Weekend and Overnight Effects


In [ ]:
# --- 8. Weekend Effect in Crypto ---

#-- Creating a new column for storing the day
btc['Day'] = btc.index.day_name()
eth['Day'] = eth.index.day_name()
sp500['Day'] = sp500.index.day_name()

#-- Calculating the average return for each day
weekend_returns_btc = btc.groupby("Day")["Log_Returns"].mean()
weekend_returns_eth = eth.groupby("Day")["Log_Returns"].mean()
weekend_returns_sp500 = sp500.groupby("Day")["Log_Returns"].mean()

print("S&P500 Average Returns by Day of the Week:")
print(weekend_returns_sp500)
print("Bitcoin Average Returns by Day of the Week:")
print(weekend_returns_btc)
print("Ethereum Average Returns by Day of the Week:")
print(weekend_returns_eth)

In [ ]:
# Combine all the data for plotting
combined = pd.DataFrame({
    'Day': weekend_returns_btc.index.tolist() + weekend_returns_eth.index.tolist() + weekend_returns_sp500.index.tolist(),
    'Avg_Log_Return': weekend_returns_btc.tolist() + weekend_returns_eth.tolist() + weekend_returns_sp500.tolist(),
    'Asset': ['Bitcoin'] * len(weekend_returns_btc) + ['Ethereum'] * len(weekend_returns_eth) + ['S&P 500'] * len(weekend_returns_sp500)
})

# Order days of the week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
combined['Day'] = pd.Categorical(combined['Day'], categories=day_order, ordered=True)
combined.sort_values('Day', inplace=True)

# Define the desired order of assets in the plot
asset_order = ['S&P 500', 'Bitcoin', 'Ethereum']

# Update the 'Asset' column in the combined DataFrame to be categorical with the desired order
combined['Asset'] = pd.Categorical(combined['Asset'], categories=asset_order, ordered=True)
colors = ["#1f77b4", "#ff7f0e", "#DDD6B8"]  # Blue, Orange, Light sand colour

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=combined, x='Day', y='Avg_Log_Return', hue='Asset', hue_order=asset_order, palette=colors)  # Use palette argument
plt.title('Average Log Returns by Day of the Week (2017-2024)')
plt.ylabel('Average Log Return')
plt.xlabel('Day')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import kruskal
from scipy.stats import friedmanchisquare

In [ ]:
# Define windows
def period_label(hour):
    if 0 <= hour < 8:
        return 'Overnight'
    elif 14 <= hour < 21:
        return 'Daytime'
    else:
        return 'Other'

In [ ]:
df = btc.rename(columns={'Log_Returns': 'Return'})
df.index = pd.to_datetime(df.index)
df['Hour'] = df.index.hour
df['Period'] = df['Hour'].apply(period_label)
df

In [ ]:
# Group returns
overnight_returns = df[df['Period'] == 'Overnight']['Return'].dropna()
daytime_returns = df[df['Period'] == 'Daytime']['Return'].dropna()

# Kruskal-Wallis test for difference between overnight and daytime
stat, p = kruskal(overnight_returns, daytime_returns)
print(f"Kruskal-Wallis statistic: {stat}, p-value: {p}")

# For weekend effect
df['DayOfWeek'] = df.index.dayofweek
weekend_returns = df[df['DayOfWeek'] >= 5]['Return'].dropna()
weekday_returns = df[df['DayOfWeek'] < 5]['Return'].dropna()

stat, p = kruskal(weekend_returns, weekday_returns)
print(f"Kruskal-Wallis (weekend vs. weekday): {stat}, p-value: {p}")

In [ ]:
# Assume btc is your DataFrame with columns ['Log_Returns']
df = btc.rename(columns={'Log_Returns': 'Return'})
df.index = pd.to_datetime(df.index)  # Ensure the index is a datetime

# Add day of week (0=Monday, ..., 6=Sunday)
df['DayOfWeek'] = df.index.dayofweek

# Separate weekend and weekday returns
weekend_returns = df[df['DayOfWeek'] >= 5]['Return'].dropna()   # Saturday=5, Sunday=6
weekday_returns = df[df['DayOfWeek'] < 5]['Return'].dropna()    # Monday=0, ..., Friday=4

# Kruskal-Wallis test for difference between weekend and weekday returns
stat, p = kruskal(weekend_returns, weekday_returns)
print(f"Kruskal-Wallis (weekend vs. weekday): statistic={stat:.4f}, p-value={p:.4f}")

# Optionally, print mean returns for context
print("Mean Weekend Return:", weekend_returns.mean())
print("Mean Weekday Return:", weekday_returns.mean())


In [ ]:
# List of asset DataFrame names (strings)
asset_names = ['btc', 'bnb', 'sol', 'eth']
results = []

for name in asset_names:
    df = eval(name).rename(columns={'Log_Returns': 'Return'})
    df.index = pd.to_datetime(df.index)
    df['Week'] = df.index.to_period('W').astype(str)
    df['DayOfWeek'] = df.index.dayofweek
    returns_by_week = df.pivot_table(index='Week', columns='DayOfWeek', values='Return')
    returns_by_week = returns_by_week.dropna()
    if returns_by_week.shape[0] > 0:
        friedman_args = [returns_by_week[day].values for day in range(7)]
        stat, p = friedmanchisquare(*friedman_args)
    else:
        stat, p = float('nan'), float('nan')
    results.append({'Asset': name.upper(), 'Friedman Statistic': stat, 'p-value': p})

# Create DataFrame for results
results_df = pd.DataFrame(results)
results_df


- We test whether daily returns differ systematically across the seven days of the week for each crypto asset: BTC, BNB, SOL, and ETH.

- **Friedman test** is used, which is a non-parametric repeated-measures test. Here, each week acts like a block, and the test compares Monday, Tuesday, ..., Sunday returns within the same weeks.

- The null hypothesis is: **there is no statistically significant day-of-the-week effect in returns**. In other words, returns are not systematically higher or lower on specific weekdays.

- The p-values are all above 0.05:
  - **BTC**: p = 0.1791
  - **BNB**: p = 0.4222
  - **SOL**: p = 0.4874
  - **ETH**: p = 0.1525

- Therefore, we fail to reject the null hypothesis for all four assets. This means there is **no statistically significant evidence of a weekday effect** in the returns of BTC, BNB, SOL, or ETH.

- In conclusion, the results suggest that crypto returns do not display a consistent weekly seasonality pattern across weekdays, at least based on this Friedman test.